# 6.1 雷达定标模拟

**学习目标**
- 理解ZDR定标的必要性
- 掌握Birdbath和干雪法定标原理
- 观察定标误差对ZDR测量的影响

**参考文献**：Ryzhkov & Zrnic, Chapter 6, pp. 561-595

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
plt.rcParams['font.family'] = ['DejaVu Sans', 'SimHei']
plt.rcParams['axes.unicode_minus'] = False

In [ ]:
def birdbath_calibration(true_zdr_db=0.0, measurement_error=0.2):
    """
    Birdbath定标法：指向天顶时，雨滴水平分布，ZDR应为0
    """
    # 模拟测量值（加误差）
    measured_zdr = true_zdr_db + np.random.randn() * measurement_error
    
    # 偏置计算
    bias = measured_zdr - true_zdr_db
    
    # 校正后
    corrected_zdr = measured_zdr - bias
    
    return measured_zdr, corrected_zdr, bias

def dry_snow_calibration(observed_zdr_db=-0.5, temp_celsius=-10):
    """
    干雪定标法：干雪的理论ZDR约为-0.5dB
    """
    expected_zdr = -0.5  # 干雪理论值
    bias = observed_zdr_db - expected_zdr
    corrected = observed_zdr_db - bias
    return observed_zdr_db, corrected, bias

def plot_calibration():
    """可视化定标过程"""
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    
    # Birdbath方法
    n_samples = 100
    true_zdr = 0.0
    errors = [0.1, 0.3, 0.5]
    
    ax1 = axes[0, 0]
    for err in errors:
        measured = [birdbath_calibration(true_zdr, err)[0] for _ in range(n_samples)]
        ax1.hist(measured, bins=20, alpha=0.5, label=f'σ={err}dB')
    ax1.axvline(x=true_zdr, color='red', linewidth=2, label='真值 ZDR=0')
    ax1.set_xlabel('测量 ZDR (dB)', fontsize=11)
    ax1.set_ylabel('频次', fontsize=11)
    ax1.set_title('Birdbath定标：测量误差分布', fontsize=12)
    ax1.legend()
    ax1.grid(True, alpha=0.3)
    
    # 定标误差传播
    ax2 = axes[0, 1]
    true_zdr_range = np.linspace(-2, 6, 100)
    for err in errors:
        measured = true_zdr_range + np.random.randn(100) * err
        ax2.scatter(true_zdr_range, measured, s=1, alpha=0.3, label=f'σ={err}dB')
    ax2.plot(true_zdr_range, true_zdr_range, 'r-', linewidth=2, label='理想校正')
    ax2.set_xlabel('真实 ZDR (dB)', fontsize=11)
    ax2.set_ylabel('测量 ZDR (dB)', fontsize=11)
    ax2.set_title('ZDR测量误差 vs 真实值', fontsize=12)
    ax2.legend()
    ax2.grid(True, alpha=0.3)
    
    # 定标方法对比
    ax3 = axes[1, 0]
    methods = ['Birdbath', '干雪法', '内检法']
    accuracies = [0.2, 0.3, 0.5]  # dB
    colors = ['steelblue', 'coral', 'mediumseagreen']
    bars = ax3.bar(methods, accuracies, color=colors, alpha=0.7, edgecolor='black')
    ax3.set_ylabel('定标精度 (dB)', fontsize=11)
    ax3.set_title('不同定标方法的典型精度', fontsize=12)
    ax3.grid(True, alpha=0.3)
    for bar, val in zip(bars, accuracies):
        ax3.text(bar.get_x() + bar.get_width()/2, val + 0.02, f'±{val}dB', ha='center')
    
    # 校正前后对比
    ax4 = axes[1, 1]
    n_points = 50
    true_z = np.random.randn(n_points) * 10 + 40  # dBZ
    true_zdr = np.random.randn(n_points) * 1 + 2  # dB
    measured_zdr = true_zdr + np.random.randn(n_points) * 0.3 - 0.5  # 偏置+噪声
    corrected_zdr = measured_zdr + 0.5  # 校正
    ax4.scatter(true_zdr, measured_zdr, c='red', s=30, alpha=0.5, label='校正前')
    ax4.scatter(true_zdr, corrected_zdr, c='blue', s=30, alpha=0.5, label='校正后')
    ax4.plot([-2, 6], [-2, 6], 'k--', linewidth=1, label='理想线')
    ax4.set_xlabel('真实 ZDR (dB)', fontsize=11)
    ax4.set_ylabel('测量/校正 ZDR (dB)', fontsize=11)
    ax4.set_title('ZDR校正效果示意', fontsize=12)
    ax4.legend()
    ax4.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    print("\n=== 定标方法对比 ===")
    print("Birdbath: 需要晴空条件，精度最高")
    print("干雪法: 利用干雪的理论ZDR值")
    print("内检法: 利用雷达系统内部参考信号")

In [ ]:
plot_calibration()

## 参考文献

- Ryzhkov, A.V., and D.S. Zrnic, 2019: *Radar Polarimetry for Weather Observations*, Chapter 6, Springer.